# TP – Comptage de mots : Python, pseudo-MapReduce et Hadoop MapReduce

**Objectif :** comparer trois approches du comptage d'occurrences de mots dans un texte :

1. une version Python séquentielle simple ;
2. une version pseudo-MapReduce en Python (étapes *map*, *shuffle*, *reduce*) ;
3. une version distribuée avec Hadoop MapReduce.

On observe les différences en termes de structure, d'outils et de temps d'exécution.

**Environnement du TP :** Manjaro Linux, Python 3.11, Apache Hadoop 3.4.0.

## 1. Principe de Hadoop MapReduce

Hadoop MapReduce est un modèle de programmation distribué permettant de traiter de grands volumes de données. Il repose sur deux phases :

- **Map** : transformation des données en paires clé/valeur ;
- **Reduce** : agrégation des valeurs par clé.

Le calcul est orchestré par **YARN** sur des données stockées dans **HDFS**. Entre les deux phases, l'étape **shuffle** regroupe les valeurs par clé.

## 2. Préparation des données

Si le fichier `mots.txt` n'existe pas, la cellule suivante le génère avec la même distribution de mots que dans le rapport (85 mots au total), répartis aléatoirement sur plusieurs lignes.

In [ ]:
import os
import random

FICHIER = "mots.txt"

DISTRIBUTION = {
    "chat": 24, "chien": 11, "arbre": 7, "fleur": 7, "maison": 5,
    "soleil": 5, "montagne": 5, "livre": 4, "ciel": 4, "mer": 3,
    "lune": 3, "route": 3, "oiseau": 2, "jardin": 1, "voiture": 1,
}

if not os.path.exists(FICHIER):
    mots = [m for m, n in DISTRIBUTION.items() for _ in range(n)]
    random.seed(42)
    random.shuffle(mots)
    with open(FICHIER, "w", encoding="utf-8") as f:
        for i in range(0, len(mots), 8):          # 8 mots par ligne
            f.write(" ".join(mots[i:i + 8]) + "\n")
    print(f"{FICHIER} généré ({len(mots)} mots).")
else:
    print(f"{FICHIER} déjà présent, il sera utilisé tel quel.")

## 3. Version 1 : Python simple

Lecture du fichier puis comptage à l'aide d'un dictionnaire Python.

In [ ]:
def compter_occurrences(fichier):
    with open(fichier, "r", encoding="utf-8") as f:
        mots = f.read().split()

    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
    return compteur


resultats_v1 = compter_occurrences(FICHIER)
for mot, nb in sorted(resultats_v1.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

## 4. Version 2 : Pseudo-MapReduce Python

Cette version reproduit les étapes du modèle MapReduce avec trois fonctions `mapper`, `shuffle` et `reducer`, exécutées localement.

In [ ]:
from collections import defaultdict


def mapper(ligne):
    """Map : une ligne -> liste de paires (mot, 1)."""
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]


def shuffle(mapped_data):
    """Shuffle : regroupe les valeurs par clé."""
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped


def reducer(grouped_data):
    """Reduce : somme des valeurs pour chaque clé."""
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced


def pseudo_mapreduce(fichier):
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            mapped.extend(mapper(ligne))
    grouped = shuffle(mapped)
    return reducer(grouped)


resultats_v2 = pseudo_mapreduce(FICHIER)
for mot, nb in sorted(resultats_v2.items(), key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

In [ ]:
# Vérification : les deux versions Python donnent le même résultat
assert resultats_v1 == resultats_v2
print("OK : versions 1 et 2 identiques,", sum(resultats_v1.values()), "mots comptés.")

## 5. Version 3 : Hadoop MapReduce

Cette version utilise l'infrastructure Hadoop (HDFS + YARN) pour exécuter le job `wordcount` fourni dans les exemples de Hadoop.

### 5.1 Installation et démarrage (à exécuter dans un terminal)

```bash
# Installation de Java et Hadoop
sudo pacman -S jdk-openjdk
wget https://archive.apache.org/dist/hadoop/common/hadoop-3.4.0/hadoop-3.4.0.tar.gz
tar -xvzf hadoop-3.4.0.tar.gz && mv hadoop-3.4.0 ~/hadoop

# Configuration de l'environnement
echo "export HADOOP_HOME=$HOME/hadoop" >> ~/.bashrc
echo 'export PATH=$PATH:$HADOOP_HOME/bin:$HADOOP_HOME/sbin' >> ~/.bashrc
source ~/.bashrc

# Démarrage du cluster
hdfs namenode -format
start-dfs.sh
start-yarn.sh
```

### 5.2 Exécution du job

La cellule suivante lance le job **si Hadoop est disponible** sur la machine. Sinon, elle utilise la durée mesurée dans le rapport (12 s).

In [ ]:
import os
import shutil
import subprocess
import time

HADOOP_HOME = os.environ.get("HADOOP_HOME", os.path.expanduser("~/hadoop"))
HDFS_USER_DIR = "/user/" + os.environ.get("USER", "imad") + "/wordcount"
DUREE_HADOOP_RAPPORT = 12.0  # secondes, valeur mesurée dans le rapport


def sh(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True)


hadoop_dispo = shutil.which("hadoop") is not None and shutil.which("hdfs") is not None

if hadoop_dispo:
    # 1. Création du répertoire HDFS et copie des données
    sh(f"hdfs dfs -rm -r -f {HDFS_USER_DIR}")
    sh(f"hdfs dfs -mkdir -p {HDFS_USER_DIR}/input")
    sh(f"hdfs dfs -put -f {FICHIER} {HDFS_USER_DIR}/input/")

    # 2. Lancement du job MapReduce
    jar = f"{HADOOP_HOME}/share/hadoop/mapreduce/hadoop-mapreduce-examples-3.4.0.jar"
    debut = time.perf_counter()
    r = sh(f"hadoop jar {jar} wordcount {HDFS_USER_DIR}/input {HDFS_USER_DIR}/output")
    duree_hadoop = time.perf_counter() - debut
    print("Code retour :", r.returncode)

    # 3. Visualisation du résultat
    print(sh(f"hdfs dfs -cat {HDFS_USER_DIR}/output/part-r-00000").stdout)
    print(f"Durée totale : {duree_hadoop:.1f} s")
else:
    duree_hadoop = DUREE_HADOOP_RAPPORT
    print("Hadoop n'est pas disponible dans cet environnement.")
    print(f"Utilisation de la durée mesurée dans le rapport : {duree_hadoop} s")

### 5.3 Résumé de l'application YARN (rapport)

| Champ | Valeur |
|---|---|
| Application ID | `application_1761131571423_0001` |
| Name | word count |
| Application Type | MAPREDUCE |
| State / Final Status | FINISHED / SUCCEEDED |
| Started | Wed Oct 22 13:18:30 +0000 2025 |
| Finished | Wed Oct 22 13:18:42 +0000 2025 |
| Elapsed | 12 s |

## 6. Comparaison et analyse des performances

On mesure les deux versions Python avec `time.perf_counter` sur plusieurs répétitions (moyenne et minimum), car un seul passage sur un fichier aussi petit est très bruité.

In [ ]:
import statistics
import time


def mesurer(fonction, *args, repetitions=200):
    durees = []
    for _ in range(repetitions):
        t0 = time.perf_counter()
        fonction(*args)
        durees.append(time.perf_counter() - t0)
    return statistics.mean(durees), min(durees)


moy_v1, min_v1 = mesurer(compter_occurrences, FICHIER)
moy_v2, min_v2 = mesurer(pseudo_mapreduce, FICHIER)

print(f"Python simple      : moyenne {moy_v1*1000:.3f} ms | min {min_v1*1000:.3f} ms")
print(f"Pseudo-MapReduce   : moyenne {moy_v2*1000:.3f} ms | min {min_v2*1000:.3f} ms")
print(f"Hadoop MapReduce   : {duree_hadoop:.1f} s")

In [ ]:
import pandas as pd

tableau = pd.DataFrame({
    "Méthode": ["Python simple", "Pseudo-MapReduce Python", "Hadoop MapReduce"],
    "Temps (s)": [moy_v1, moy_v2, duree_hadoop],
    "Remarques": [
        "Lecture directe, très rapide sur petits fichiers.",
        "Simulation des étapes Map/Reduce, surcharge faible.",
        "Latence au démarrage, mais adapté aux gros volumes.",
    ],
})
tableau["Temps (s)"] = tableau["Temps (s)"].map(lambda x: f"{x:.6f}" if x < 1 else f"{x:.1f}")
tableau

In [ ]:
import matplotlib.pyplot as plt

noms = ["Python simple", "Pseudo-MapReduce", "Hadoop MapReduce"]
temps = [moy_v1, moy_v2, duree_hadoop]

fig, ax = plt.subplots(figsize=(7, 4))
barres = ax.bar(noms, temps, color=["#4c78a8", "#f58518", "#54a24b"])
ax.set_yscale("log")
ax.set_ylabel("Temps d'exécution (s, échelle log)")
ax.set_title("Comparaison des temps d'exécution")
for b, t in zip(barres, temps):
    ax.text(b.get_x() + b.get_width() / 2, t, f"{t:.4g} s", ha="center", va="bottom")
plt.tight_layout()
plt.show()

### 6.1 Analyse

- Les versions Python sont **beaucoup plus rapides** sur de petits fichiers : tout se fait en mémoire, sans démarrage de processus.
- La version pseudo-MapReduce est légèrement plus lente que la version simple (création des paires `(mot, 1)` puis regroupement), mais la **surcharge reste faible**.
- Hadoop introduit une **latence importante** liée au démarrage des processus distribués (JVM, YARN, lecture/écriture HDFS), de l'ordre de plusieurs secondes quelle que soit la taille des données.
- En revanche, Hadoop devient plus performant dès que la taille du jeu de données dépasse **plusieurs gigaoctets**, grâce à sa capacité de parallélisation sur plusieurs nœuds.

## 7. Conclusion

Ce travail illustre le modèle MapReduce à travers trois approches complémentaires : la **simplicité** du traitement local, la **modularité** du modèle simulé, et la **puissance** du traitement parallèle offert par Hadoop. Ces trois approches forment une progression naturelle : de la compréhension du concept à sa mise en œuvre réelle dans un environnement distribué.